# Recruiting Router Eval

This notebook evaluates the current recruiting chatbot against `Database/sms_conversations.json`.

Process:
1. Build eval examples from every candidate turn.
2. Use the next recruiter turn label as the expected router action.
3. Run the real backend path through `process_candidate_turn()`.
4. Measure label accuracy and confusion matrix.
5. When expected and predicted labels are both `schedule`, compare expected recruiter slot text with returned SQL-backed slots.

Opening recruiter greetings are not routed. They only appear in history for the first candidate response.

In [1]:
from pathlib import Path
import json
import re
import sys

import pandas as pd
from sklearn.metrics import accuracy_score, confusion_matrix

PROJECT_ROOT = Path.cwd() # set path with protecting against running from other directories
if not (PROJECT_ROOT / "Database" / "sms_conversations.json").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

sys.path.insert(0, str(PROJECT_ROOT))

from app.modules.conversation_service import CandidateTurnInput, process_candidate_turn

DATA_PATH = PROJECT_ROOT / "Database" / "sms_conversations.json"
RESULTS_PATH = PROJECT_ROOT / "tests" / "eval_results.csv"
ROLE = "Python Developer"

print(PROJECT_ROOT)
print(DATA_PATH)

C:\GenAI Final Project
C:\GenAI Final Project\Database\sms_conversations.json


## Build Eval Examples

The JSON labels live on recruiter turns. For router evals, each candidate message is paired with the immediately following recruiter turn. The candidate text is the input, previous turns are history, and the next recruiter label is the expected action.

In [2]:
def build_examples(data):
    examples = []

    for conversation in data:
        turns = conversation["turns"]
        for idx, turn in enumerate(turns):
            if turn["speaker"] != "candidate":
                continue

            if idx + 1 >= len(turns):
                continue

            next_turn = turns[idx + 1]
            if next_turn["speaker"] != "recruiter" or not next_turn.get("label"):
                continue

            examples.append(
                {
                    "conversation_id": conversation["conversation_id"],
                    "turn_id": turn["turn_id"],
                    "message": turn["text"],
                    "history": [previous["text"] for previous in turns[:idx]],
                    "expected_action": next_turn["label"],
                    "expected_reply": next_turn["text"],
                    "reference_datetime_utc": turn.get("timestamp_utc"),
                }
            )

    return examples


with DATA_PATH.open("r", encoding="utf-8") as file:
    conversations = json.load(file)

examples = build_examples(conversations)
examples_df = pd.DataFrame(examples)

print(f"conversations: {len(conversations)}")
print(f"router eval examples: {len(examples)}")
display(examples_df["expected_action"].value_counts().rename_axis("expected_action").reset_index(name="count"))
display(examples_df.head())

conversations: 15
router eval examples: 44


,expected_action,count
0,schedule,19
1,end,15
2,continue,10


,conversation_id,turn_id,message,history,expected_action,expected_reply,reference_datetime_utc
0,1,2,I've been using Python professionally for five...,[Thanks for applying to our Python Developer o...,schedule,Our engineering manager can interview you Wedn...,2024-04-03T15:13:19Z
1,1,4,I can't at that time—I'm busy.,[Thanks for applying to our Python Developer o...,schedule,No problem. How about Thursday at 4 PM instead?,2024-04-03T15:15:33Z
2,1,6,Monday at 3 PM is good.,[Thanks for applying to our Python Developer o...,end,"Great, your interview is confirmed. You'll rec...",2024-04-03T15:17:03Z
3,2,2,I have three years' experience with Django and...,"[Hi, thanks for submitting your application fo...",continue,Do you have any questions of your own?,2024-04-30T11:19:48Z
4,2,4,Could you share more about the company's cloud...,"[Hi, thanks for submitting your application fo...",continue,We currently deploy to AWS using Docker and ECS.,2024-04-30T11:23:25Z


## Slot Normalization

The JSON has human-written recruiter slot text, while the app returns SQL-backed formatted slots. This comparison is intentionally simple: extract weekday/time pairs when available, otherwise weekday-only mentions such as `next Thursday`.

In [3]:
WEEKDAYS = "Monday|Tuesday|Wednesday|Thursday|Friday|Saturday|Sunday"


def normalize_time(hour_text, minute_text, meridiem):
    hour = int(hour_text)
    minute = int(minute_text or 0)
    if meridiem:
        meridiem = meridiem.lower()
        if meridiem == "pm" and hour != 12:
            hour += 12
        if meridiem == "am" and hour == 12:
            hour = 0
    return f"{hour:02d}:{minute:02d}"


def extract_slot_keys(text):
    if not text:
        return set()

    normalized = text.replace("\u202f", " ").replace("\xa0", " ")
    keys = set()

    weekday_time_pattern = re.compile(
        rf"(?P<weekday>{WEEKDAYS})(?:,?\s+\d{{4}}-\d{{2}}-\d{{2}})?(?:\s+at)?\s+"
        r"(?P<hour>\d{1,2})(?::(?P<minute>\d{2}))?\s*(?P<meridiem>AM|PM|am|pm)?",
        re.IGNORECASE,
    )
    for match in weekday_time_pattern.finditer(normalized):
        weekday = match.group("weekday").lower()
        time_text = normalize_time(match.group("hour"), match.group("minute"), match.group("meridiem"))
        keys.add(f"{weekday}|{time_text}")

    if not keys:
        weekday_pattern = re.compile(rf"(?:next\s+)?(?P<weekday>{WEEKDAYS})", re.IGNORECASE)
        keys.update(match.group("weekday").lower() for match in weekday_pattern.finditer(normalized))

    return keys


def slot_match(expected_reply, predicted_slots):
    expected_keys = extract_slot_keys(expected_reply)
    predicted_keys = set()
    for slot in predicted_slots or []:
        predicted_keys.update(extract_slot_keys(slot))

    if not expected_keys:
        return None, expected_keys, predicted_keys

    return bool(expected_keys & predicted_keys), expected_keys, predicted_keys


print(extract_slot_keys("Our manager can interview Wednesday at 10 AM or Thursday at 2 PM."))
print(extract_slot_keys("Tuesday, 2024-01-02 at 09:00"))

{'wednesday|10:00', 'thursday|14:00'}
{'tuesday|09:00'}


## Run Current Backend

This calls the current project code, including the main router, advisors, OpenAI structured outputs, and SQL schedule lookup. Set `MAX_EXAMPLES` to a small number for quick debugging; leave it as `None` for the full dataset.

In [4]:
MAX_EXAMPLES = None

rows = []
eval_examples = examples[:MAX_EXAMPLES] if MAX_EXAMPLES else examples

for index, example in enumerate(eval_examples, start=1):
    print(f"{index}/{len(eval_examples)} conversation={example['conversation_id']} turn={example['turn_id']}")

    result = process_candidate_turn(
        CandidateTurnInput(
            message=example["message"],
            role=ROLE,
            history=example["history"],
            first_name="Eval",
            last_name="Candidate",
            reference_datetime_utc=example["reference_datetime_utc"]
        )
    )

    label_match = result.action == example["expected_action"]
    schedule_slot_match = None
    expected_slot_keys = set()
    predicted_slot_keys = set()

    if label_match and result.action == "schedule" and example["expected_action"] == "schedule":
        schedule_slot_match, expected_slot_keys, predicted_slot_keys = slot_match(
            example["expected_reply"], result.slots
        )

    rows.append(
        {
            "conversation_id": example["conversation_id"],
            "turn_id": example["turn_id"],
            "message": example["message"],
            "expected_action": example["expected_action"],
            "predicted_action": result.action,
            "label_match": label_match,
            "expected_reply": example["expected_reply"],
            "assistant_message": result.assistant_message,
            "predicted_slots": result.slots or [],
            "expected_slot_keys": sorted(expected_slot_keys),
            "predicted_slot_keys": sorted(predicted_slot_keys),
            "slot_match": schedule_slot_match,
        }
    )

results_df = pd.DataFrame(rows)
results_df.to_csv(RESULTS_PATH, index=False)
print(f"saved: {RESULTS_PATH}")
display(results_df.head())

1/44 conversation=1 turn=2


2/44 conversation=1 turn=4


3/44 conversation=1 turn=6


4/44 conversation=2 turn=2


5/44 conversation=2 turn=4


...

42/44 conversation=14 turn=6


43/44 conversation=15 turn=2


44/44 conversation=15 turn=4


saved: C:\GenAI Final Project\tests\eval_results.csv


,conversation_id,turn_id,message,expected_action,predicted_action,label_match,expected_reply,assistant_message,predicted_slots,expected_slot_keys,predicted_slot_keys,slot_match
0,1,2,I've been using Python professionally for five...,schedule,schedule,True,Our engineering manager can interview you Wedn...,"Thanks for sharing your experience, Eval! It s...","[Wednesday, 2024-04-03 at 10:00, Wednesday, 20...","[thursday|14:00, wednesday|10:00]","[wednesday|10:00, wednesday|13:00, wednesday|1...",True
1,1,4,I can't at that time—I'm busy.,schedule,schedule,True,No problem. How about Thursday at 4 PM instead?,"No problem, Eval! Let's find a time that works...","[Wednesday, 2024-04-03 at 10:00, Wednesday, 20...",[thursday|16:00],"[wednesday|10:00, wednesday|13:00, wednesday|1...",False
2,1,6,Monday at 3 PM is good.,end,schedule,False,"Great, your interview is confirmed. You'll rec...","Thanks for your response, Eval! Unfortunately,...","[Tuesday, 2024-04-09 at 11:00, Tuesday, 2024-0...",[],[],None
3,2,2,I have three years' experience with Django and...,continue,schedule,False,Do you have any questions of your own?,"Thanks for sharing your experience, Eval! It s...","[Tuesday, 2024-04-30 at 09:00, Tuesday, 2024-0...",[],[],None
4,2,4,Could you share more about the company's cloud...,continue,continue,True,We currently deploy to AWS using Docker and ECS.,"That's a great question, Eval! I currently don...",[],[],[],None


## Report

In [5]:
labels = ["continue", "schedule", "end"]
accuracy = accuracy_score(results_df["expected_action"], results_df["predicted_action"])
matrix = confusion_matrix(results_df["expected_action"], results_df["predicted_action"], labels=labels)
confusion_df = pd.DataFrame(matrix, index=[f"expected_{label}" for label in labels], columns=[f"predicted_{label}" for label in labels])

schedule_rows = results_df[
    (results_df["expected_action"] == "schedule")
    & (results_df["predicted_action"] == "schedule")
]
slot_rows_with_expected = schedule_rows[schedule_rows["slot_match"].notna()]
slot_accuracy = None
if len(slot_rows_with_expected) > 0:
    slot_accuracy = slot_rows_with_expected["slot_match"].mean()

print(f"examples: {len(results_df)}")
print(f"label accuracy: {accuracy:.3f}")
print(f"correct schedule labels: {len(schedule_rows)}")
if slot_accuracy is None:
    print("slot match accuracy: n/a")
else:
    print(f"slot match accuracy on correct schedule rows with expected slots: {slot_accuracy:.3f}")

display(confusion_df)
display(results_df[~results_df["label_match"]][["conversation_id", "turn_id", "message", "expected_action", "predicted_action", "assistant_message"]])
display(slot_rows_with_expected[["conversation_id", "turn_id", "expected_slot_keys", "predicted_slot_keys", "slot_match"]])

examples: 44
label accuracy: 0.773
correct schedule labels: 17
slot match accuracy on correct schedule rows with expected slots: 0.235


,predicted_continue,predicted_schedule,predicted_end
expected_continue,5,5,0
expected_schedule,1,17,1
expected_end,0,3,12


,conversation_id,turn_id,message,expected_action,predicted_action,assistant_message
2,1,6,Monday at 3 PM is good.,end,schedule,"Thanks for your response, Eval! Unfortunately,..."
3,2,2,I have three years' experience with Django and...,continue,schedule,"Thanks for sharing your experience, Eval! It s..."
7,3,2,"Sure, I have four years of Python experience a...",continue,schedule,"Thanks for sharing your experience, Eval! It s..."
9,3,6,"I would like to set an appointment, does Monda...",end,schedule,"Thanks for your interest, Eval! Unfortunately,..."
10,4,2,I've been using Python professionally for five...,continue,schedule,"Thanks for sharing your experience, Eval! It s..."
22,8,6,"I'm unavailable at that time, as I have other ...",schedule,end,"Thanks for letting me know, Eval. Feel free to..."
28,10,4,"I've worked a bit with AWS, but I don't have e...",schedule,continue,"Thanks for sharing, Eval! It's great to hear a..."
30,11,2,I have three years' experience with Pyhon and ...,continue,schedule,"Thanks for sharing your experience, Eval! It s..."
31,11,4,"I've worked a bit with AWS, mainly for deployi...",continue,schedule,"Thanks for sharing, Eval! Your experience with..."
36,12,6,Wednesday at 10 AM works for me.,end,schedule,"Thanks for your response, Eval! Wednesday, 202..."


,conversation_id,turn_id,expected_slot_keys,predicted_slot_keys,slot_match
0,1,2,"[thursday|14:00, wednesday|10:00]","[wednesday|10:00, wednesday|13:00, wednesday|1...",True
1,1,4,[thursday|16:00],"[wednesday|10:00, wednesday|13:00, wednesday|1...",False
5,2,6,[tuesday|10:00],"[tuesday|09:00, tuesday|13:00, tuesday|14:00]",False
13,5,2,"[thursday|14:00, wednesday|10:00]","[friday|10:00, friday|11:00, friday|13:00]",False
15,6,2,"[friday|11:00, monday|09:00]","[tuesday|13:00, tuesday|14:00, tuesday|16:00]",False
17,7,2,"[monday|15:00, tuesday|11:00]","[friday|12:00, friday|15:00, friday|17:00]",False
18,7,4,[tuesday|10:00],"[friday|12:00, friday|15:00, friday|17:00]",False
20,8,2,"[thursday|14:00, wednesday|10:00]","[wednesday|10:00, wednesday|13:00, wednesday|1...",True
21,8,4,[tuesday|10:00],"[wednesday|10:00, wednesday|13:00, wednesday|1...",False
24,9,2,"[thursday|14:00, wednesday|10:00]","[wednesday|09:00, wednesday|10:00, wednesday|1...",True
